In [ ]:
import os
import sys
import json
import glob

import pandas as pd
from openpyxl import load_workbook

# Make the shared cleaning package (backend/cleaning/) importable from this notebook
sys.path.insert(0, os.path.abspath(os.path.join("..", "backend")))

# NOTE: cleaning.master_json is deprecated (see that file) and not imported here.
from cleaning import io_utils, census_transform, proportions, recipes, pipeline

### Basic Utilities — now in `backend/cleaning/io_utils.py`

### Census Dataset Utilities — now in `backend/cleaning/census_transform.py`

### Dataset processing pipeline — now in `backend/cleaning/pipeline.py`

### load_dataset_config, process_census_dataset, load_and_process_all_datasets now live in backend/cleaning/pipeline.py

In [ ]:
path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/")
notes_wb_path = os.path.join(path_head, "Metrics/Notes/JPL_CCSVI_all_fields.xlsx")

In [ ]:
notes_wb = load_workbook(notes_wb_path)
sheet = notes_wb.active

In [ ]:
file_blocks = {}
current_file_name = None
cols_not_to_drop = ["Geography", "Geographic Area Name"] 

# Iterate through rows to find blocks for each csv file
for row in sheet.iter_rows(min_row=1, max_col=3, values_only=False):
    cell_value = row[0].value
    is_bold = row[0].font.bold if row[0].font else False

    # Detect file block by bold file name
    if is_bold and cell_value:
        original_name = cell_value.strip()
        current_file_name = io_utils.to_snake_case(original_name)
        file_blocks[current_file_name] = {
            'original_name': original_name,
            'cols_to_drop': [],
            'code_to_alias_column_mappings': {},
            'original_file_path': '',
            'centralized_file_dir': '' 
        }
        continue

    # Check for columns with a "Subfield to keep" value
    if current_file_name and any(col.value for col in row):

        col_name = row[0].value
        subfield_value = row[1].value
        # Track columns with no value in the subfields to keep column
        if col_name and not subfield_value:
            if col_name not in cols_not_to_drop:
                cleaned_col_name = col_name.strip().strip('\'"')
                cleaned_col_name = ' '.join(cleaned_col_name.split())
                file_blocks[current_file_name]['cols_to_drop'].append(cleaned_col_name)

In [ ]:
list(file_blocks.items())

In [ ]:
# exposures_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures")
central_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/JPL")
cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data-TEST")
json_dir_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/census-jsons")

In [ ]:
config_path = "../backend/cleaning/config/census_datasets_config.json"
base_path = "~/Desktop/Nextcloud/SCOVI Project/Metrics/"

In [ ]:
datasets = pipeline.load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head)

In [ ]:
# Runs every dataset in the config through its recipe and writes one CSV per output into cleaned_path_head.
# The per-dataset cells below only look at the tables; this is what actually writes them.
cleaned = pipeline.run_full_cleaning(config_path, base_path, notes_wb_path, central_path_head, output_dir=cleaned_path_head)
sorted(cleaned)

In [ ]:
# datasets.keys()

### Age of Structure

##### JPL Notes Cleaning

In [ ]:
age_of_structure_df = datasets['age_of_structure']
age_of_structure_df.head(2)

In [ ]:
cleaned_age_of_structure_df = recipes.recipe_age_of_structure(age_of_structure_df)
cleaned_age_of_structure_df.head(2)

### Aggregate number of vehicles

In [ ]:
aggregate_vehicles_df = datasets['aggregate_vehicles']
cleaned_aggregate_vehicles_df = recipes.merge_tenure_households(aggregate_vehicles_df, datasets['tenure'])
cleaned_aggregate_vehicles_df.head(2)

### Health insurance

##### JPL Notes Cleaning

In [ ]:
health_insurance_df = datasets['health_insurance']
health_insurance_df.head(2)

In [ ]:
cleaned_health_insurance_df = recipes.recipe_health_insurance(health_insurance_df)
cleaned_health_insurance_df.head(2)

### Households with a computer

In [ ]:
households_w_computer_df = datasets['households_w_computer']
households_w_computer_df.head(2)

### Internet subscription

In [ ]:
internet_subscription_df = datasets['internet_subscription']
internet_subscription_df.head(2)

### Limited English speaking

##### JPL Notes Cleaning

In [ ]:
limited_english_speaking_df = datasets['limited_english_speaking']
limited_english_speaking_df.head(2)

In [ ]:
cleaned_limited_english_speaking_df = recipes.recipe_limited_english_speaking(limited_english_speaking_df)
cleaned_limited_english_speaking_df.head(2)

### Living Arrangements

##### JPL Notes Cleaning

In [ ]:
living_arrangements_df = datasets['living_arrangements']
living_arrangements_df.head(2)

In [ ]:
cleaned_living_arrangements_df = recipes.recipe_living_arrangements(living_arrangements_df)
cleaned_living_arrangements_df.head(2)

### Income share of FPL

##### JPL Notes Cleaning

In [ ]:
income_share_of_fpl_df = datasets['income_share_of_fpl']
income_share_of_fpl_df.head(2)

In [ ]:
cleaned_fpl_df = recipes.recipe_income_share_of_fpl(income_share_of_fpl_df)
cleaned_fpl_df.head(3)

### Persons under 5 & 65

##### JPL Notes Cleaning

In [ ]:
person_under_5_65_df = datasets['person_under_5_65']
person_under_5_65_df.head(2)

In [ ]:
person_under_5_65_outputs = recipes.recipe_person_under_5_65(person_under_5_65_df)
person_under_5_65_outputs['genders'].head(2)

In [ ]:
person_under_5_65_outputs['person_under_5_65_males'].head(2)

In [ ]:
person_under_5_65_outputs['person_under_5_65_females'].head(2)

### Population in group quarters

In [ ]:
population_group_quarters_df = datasets['population_group_quarters']
population_group_quarters_df.head(2)

### Race origin

In [ ]:
race_origin_df = datasets['race_origin']
race_origin_df.head(2)

### Tenure

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

### 2022 Census Hawaiian Homelands

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
# export_census_csv(tenure_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### JPL Notes Cleaning

In [ ]:
hawaiian_homelands_df = datasets['2022_census_hawaiian_homelands']
hawaiian_homelands_df.head(2)

In [ ]:
cleaned_hawaiian_homelands_df = recipes.recipe_hawaiian_homelands(hawaiian_homelands_df)
cleaned_hawaiian_homelands_df.head(2)

## Add Proportions to All Datasets ====================================

In [ ]:
# Block groups population from 2020 Census
block_groups_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/2020_Census_Block_Groups_Stripped.geojson")
hawaiian_homelands_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/Census_Hawaiian_Homelands_hhl10_Stripped.geojson")

block_group_populations = proportions.load_block_group_populations(block_groups_geojson_path)
hawaiian_homelands_populations = proportions.load_hawaiian_homelands_populations(hawaiian_homelands_geojson_path)

total_population_block_groups = sum(block_group_populations.values())
total_population_hawaiian_homelands = sum(hawaiian_homelands_populations.values())
total_population = total_population_block_groups + total_population_hawaiian_homelands

print(f"Total population (block groups): {total_population_block_groups}")
print(f"Total population (Hawaiian homelands): {total_population_hawaiian_homelands}")
print(f"Total population (combined): {total_population}")

## ============================================================

In [ ]:
# import glob

# csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

# # Process each CSV file
# for csv_file in csv_files:
#     print(f"Processing {csv_file}...")
#     try:
#         census_csv_to_json(csv_file, json_dir_path)
#     except Exception as e:
#         print(f"Error processing {csv_file}: {e}")

In [ ]:
# Uses the real pipeline function directly, so this can never drift out of sync with backend/cleaning/pipeline.py again
pipeline.add_percentages_to_directory(
    cleaned_path_head,
    config_path,
    base_path,
    block_group_populations,
    hawaiian_homelands_populations,
)

##### Percentages over 100%

In [ ]:
proportions.check_percentages_over_100(cleaned_path_head)

##### ACS population estimates vs the 2020 Census baseline

In [ ]:
proportions.check_population_source_mismatch(
    os.path.join(central_path_head, "person_under_5_65"),
    block_group_populations,
)

##### If we used each dataset's own ACS col total instead of Census pop20

In [ ]:
# Population-based datasets only — the household/housing-unit datasets already use their
# own dataset's total (via percent_denominator in the config), so there's nothing to compare there.
population_based_datasets = [
    ("genders.csv", "person_under_5_65"),
    ("person_under_5_65_males.csv", "person_under_5_65"),
    ("person_under_5_65_females.csv", "person_under_5_65"),
    ("race_origin.csv", "race_origin"),
    ("health_insurance.csv", "health_insurance"),
    ("living_arrangements.csv", "living_arrangements"),
    ("income_share_of_fpl.csv", "income_share_of_fpl"),
    ("population_group_quarters.csv", "population_group_quarters"),
]

for csv_filename, raw_dir_name in population_based_datasets:
    proportions.compare_population_denominators(
        os.path.join(cleaned_path_head, csv_filename),
        os.path.join(central_path_head, raw_dir_name),
    )

## Leaflet JSON 

In [ ]:
# DEPRECATED: backend/cleaning/master_json.py (census_csvs_to_master_json, clean_column_name)
# built census_metrics_by_block_group.json for the old census_metrics table, which was
# migrated to the current normalized schema and dropped by backend/ingest/migrate_schema.py.
# Nothing reads that JSON anymore — see ADDING_DATASETS_GUIDE.md §2 for the current path

# generate_dataset_params: replaced by per-map classification mode UI control.
# Pre-computing quantile thresholds and color schemes server-side is no longer
# needed — the frontend computes the color scale at render time based on the
# user's chosen classification mode.

In [ ]:
# DEPRECATED — see the note above. Nothing consumes this JSON anymore; kept commented for reference.
# json_output_path = os.path.join(json_dir_path,"census_metrics_by_block_group.json") 
# metrics = master_json.census_csvs_to_master_json(cleaned_path_head, json_output_path, block_group_populations, hawaiian_homelands_populations)

In [ ]:
# census_datasets_config.json generation removed — dataset metadata (labels,
# hawaiian_homelands flag, mappable columns) will be stored directly in the DB.
# Classification mode and color schemes are now per-map UI controls.
#
# json_output_path = os.path.join(json_dir_path, "census_datasets_config.json")
#
# dataset_params = generate_dataset_params(
#     cleaned_path_head,
#     json_output_path,
#     block_group_populations,
#     hawaiian_homelands_populations
# )

### Debugging the math ==================

In [ ]:
import csv
import glob
import re

import numpy as np

def read_raw_dataset(dataset_dir_name):
    path = glob.glob(os.path.join(central_path_head, dataset_dir_name, "*.csv"))[0]
    names = list(csv.reader(open(path, encoding="utf-8-sig")))[1]
    df = pd.read_csv(path, skiprows=[0], na_values=["", "-", "**", "null", "(X)"], encoding="utf-8-sig")
    df.columns = names
    return df

def read_cleaned(filename):
    return pd.read_csv(os.path.join(cleaned_path_head, filename), na_values=["", "-", "**", "null"])

data_dict = {
    "2022_census_hawaiian_homelands.csv": "2022_census_hawaiian_homelands",
    "age_of_structure.csv": "age_of_structure",
    "aggregate_vehicles.csv": "aggregate_vehicles",
    "family_type_by_children.csv": "family_type_by_children",
    "genders.csv": "person_under_5_65",
    "health_insurance.csv": "health_insurance",
    "households_w_computer.csv": "households_w_computer",
    "income_share_of_fpl.csv": "income_share_of_fpl",
    "internet_subscription.csv": "internet_subscription",
    "limited_english_speaking.csv": "limited_english_speaking",
    "living_arrangements.csv": "living_arrangements",
    "person_under_5_65_females.csv": "person_under_5_65",
    "person_under_5_65_males.csv": "person_under_5_65",
    "population_group_quarters.csv": "population_group_quarters",
    "race_origin.csv": "race_origin",
    "tenure.csv": "tenure",
    "tenure_by_occupants_per_room.csv": "tenure_by_occupants_per_room",
}

# {output file name: percent_denominator}, straight from the config
audit_denominator_map = {
    output["name"]: output["percent_denominator"]
    for output in pipeline.dataset_outputs(pipeline.load_dataset_config(config_path, base_path))
}

### Check cleaning code

In [ ]:
# recipes_path = os.path.join("..", "backend", "cleaning", "recipes.py")
# print("Positional joins in recipes.py:\n")
# for line_number, line in enumerate(open(recipes_path).read().splitlines(), 1):
#     if "concat" in line:
#         print(f"recipes.py:{line_number}: {line.strip()}")


# # Two tables can only be joined by row position if they hae the same places in same order
# def check_positional_merge(label, left_dir, right_dir):
#     left, right = read_raw_dataset(left_dir), read_raw_dataset(right_dir)
#     same_order = list(left["Geography"]) == list(right["Geography"])
#     print(f"{len(left)} vs {len(right)} rows, same places in same order: {same_order}")


# check_positional_merge("aggregate_vehicles + tenure  (merge_tenure_households)",
#                        "aggregate_vehicles", "tenure")

# check_positional_merge("income_share_of_fpl + Homelands  (merge_hawaiian_homelands_poverty)",
#                        "income_share_of_fpl", "2022_census_hawaiian_homelands")

# # Hawaiian Homelands

In [ ]:
hh_check = read_raw_dataset("2022_census_hawaiian_homelands")

# AGE rows add to 100, so they are prob percentages, not people
hh_check.filter(regex=r"^Estimate!!Total!!Total population(!!(AGE|SEX)!!.*)?$").head(3).T

## Everything uses 2020 census BG pops, maybe they should use their own total cols

Where `percent_denominator` is `null`, the pipeline falls back to the 2020 Census population.

In [ ]:
UNIVERSE_COLUMNS = ["Estimate!!Total:", "Estimate!!Total!!Total population"]

rows = []
for cleaned_name, raw_dir in sorted(data_dict.items()):
    cleaned_df, raw_df = read_cleaned(cleaned_name), read_raw_dataset(raw_dir)
    alias = os.path.splitext(cleaned_name)[0]
    universe_col = next((c for c in UNIVERSE_COLUMNS if c in raw_df.columns), None)
    if universe_col is None or "Census_Population" not in cleaned_df.columns:
        continue

    ratio = (raw_df[universe_col].reset_index(drop=True)
             / cleaned_df["Census_Population"].reset_index(drop=True))
    ratio = ratio.replace([np.inf, -np.inf], np.nan)

    configured = audit_denominator_map.get(alias)
    rows.append({
        "dataset": alias,
        "divides by": configured if configured else "2020 Census population",
        "own universe / 2020 pop": round(ratio.median(), 3),
    })

universe_check = pd.DataFrame(rows)

# Classify by how close each dataset's ratio actually sits to the population cluster, rather than
# by whether it happens to have a percent_denominator configured -- a dataset can divide by its own
# total and still be measuring everyone (health_insurance, race_origin, etc.), so "has a configured
# denominator" was never the same question as "is this a household count"
is_population_sized = universe_check["own universe / 2020 pop"] > 0.6
baseline = universe_check.loc[is_population_sized, "own universe / 2020 pop"].median()
gap = (universe_check["own universe / 2020 pop"] - baseline).abs()

universe_check["verdict"] = np.select(
    [~is_population_sized, gap > 0.05, gap > 0.02],
    ["smaller universe (household/housing-unit count)", "way off",
     "slightly off"],
    default="consistent with a whole-population table")

print(f"whole-population baseline (median of the population-sized datasets): {baseline}")
display(universe_check)


**`2022_census_hawaiian_homelands`** 2010 population
  being used against 2022 data.

### =====================================

### Are Hawaiian Homelands metrics already percentages

In [ ]:
hh_raw = read_raw_dataset("2022_census_hawaiian_homelands")

# Counts are always whole numbers, so any column carrying a decimal is not a count —
# it's a percentage, median or average, and must not be turned into a percentage again
totals = hh_raw.filter(regex=r"^Estimate!!Total!!")
is_count = ~(totals % 1 != 0).any()

print(f"{is_count.sum()} of {len(is_count)} columns contain whole number values\n")
for col in totals.columns[is_count]:
    print(f"   max {totals[col].max():>6.0f}   {col.replace('Estimate!!Total!!', '')[:68]}")

The other 47 columns all carry decimals, so they are unlikely to be measuring counts.
That includes every age, sex, race, language and poverty column the pipeline reads.

### fpl dataset only surveys a subset of the block groups, using 2020 bg pop data is misleading as well

In [ ]:
over_100 = cleaned_fpl_df[cleaned_fpl_df["Estimate!!Total: (%)"] > 100]
print("rows above 100%:", len(over_100), "of", len(cleaned_fpl_df))
print("median:", over_100["Estimate!!Total: (%)"].median(), "  highest:", over_100["Estimate!!Total: (%)"].max())
display(over_100[["Geography", "Census_Population", "Estimate!!Total:", "Estimate!!Total: (%)"]].head())

### Health insurance only surveys a subset of the block groups, using 2020 bg pop data is misleading as well

The health insurance table deliberately excludes active-duty military and people living in
institutions. Dividing by the full population therefore understates the uninsured share
everywhere.

In [ ]:
health_raw, health_cleaned = read_raw_dataset("health_insurance"), read_cleaned("health_insurance.csv")
universe_share = health_raw["Estimate!!Total:"] / health_cleaned["Census_Population"]

print("   median:", round(universe_share.median(), 4))
print("   smaller than total population in",
      (health_raw["Estimate!!Total:"] < health_cleaned["Census_Population"]).sum(),
      "of", len(health_cleaned), "block groups")

The percentage step knows to skip `Median` columns, but not `Average`, `Mean`, `Rate` or `Ratio`.

The only expected hits are `aggregate_vehicles` (known), the one Median income column (correctly
skipped), and the misplaced poverty columns from Finding 2. **The filter is a hardcoded word
list**, so it won't catch a future dataset with an `Average` column — the unused `household_income`
folder already sitting in `central_path_head` is the obvious next candidate.

In [ ]:
suspicious = re.compile(r"average|mean|rate|ratio|aggregate|per capita|median|index|percent", re.I)

for csv_file in sorted(glob.glob(os.path.join(cleaned_path_head, "*.csv"))):
    for col in pd.read_csv(csv_file, nrows=0).columns:
        if suspicious.search(col) and not col.startswith("Margin"):
            print(os.path.basename(csv_file), "::", col)